# CSV & Text Files

## Specifying Column Data Types

In [2]:
# Indicate data type either for the whole DataFrame OR for individual columns
import numpy as np
import pandas as pd
from io import StringIO

data = "a,b,c,d\n1,2,3,4\n5,6,7,8\n9,10,11"
print(data)

a,b,c,d
1,2,3,4
5,6,7,8
9,10,11


In [3]:
df = pd.read_csv(StringIO(data), dtype=object)
df

,a,b,c,d
0,1,2,3,4
1,5,6,7,8
2,9,10,11,NaN


In [4]:
# Access the value in column "a" with index 0
df["a"][0]

'1'

In [5]:
df = pd.read_csv(StringIO(data), dtype={"b": object, "c": np.float64, "d": "Int64"})
df.dtypes

a      int64
b     object
c    float64
d      Int64
dtype: object

In [6]:
data = "col_1\n1\n2\n'A'\n4.22"
df = pd.read_csv(StringIO(data), converters={"col_1": str})
df

,col_1
0,1
1,2
2,'A'
3,4.22


In [7]:
df["col_1"].apply(type).value_counts()

col_1
<class 'str'>    4
Name: count, dtype: int64

In [8]:
# To coerce the dtypes after reading the data:
df2 = pd.read_csv(StringIO(data))
df2.col_1 = pd.to_numeric(df2.col_1, errors="coerce")
df2

,col_1
0,1.00
1,2.00
2,NaN
3,4.22


In [9]:
df2.col_1.apply(type).value_counts()

col_1
<class 'float'>    4
Name: count, dtype: int64

In [10]:
# Sometimes, we can end up with columns with mixed data types
col_1 = list(range(500_000)) + ["a", "b"] + list(range(500_000))
df = pd.DataFrame({"col_1": col_1})
df.to_csv("foo.csv")

In [11]:
mixed_df = pd.read_csv("foo.csv")
mixed_df.col_1.apply(type).value_counts()

C:\Users\nkmil\AppData\Local\Temp\ipykernel_11656\2305913437.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  mixed_df = pd.read_csv("foo.csv")


col_1
<class 'int'>    737858
<class 'str'>    262144
Name: count, dtype: int64

In [12]:
mixed_df.col_1.dtype

dtype('O')

In [13]:
# dtype_backend="numpy_nullable": results in nullable data types for every column
data = """a,b,c,d,e,f,g,h,i,j
        1,2.5,True,a,,,,,12-31-2019,
        3,4.5,False,b,6,7.5,True,a,12-31-2019
"""

df = pd.read_csv(StringIO(data), dtype_backend="numpy_nullable", parse_dates=["i"])
df

,a,b,c,d,e,f,g,h,i,j
0,1,2.5,True,a,<NA>,<NA>,<NA>,<NA>,2019-12-31,<NA>
1,3,4.5,False,b,6,7.5,True,a,2019-12-31,<NA>


In [14]:
# NOTE: Column i's data type is properly recognized as datetime64[ns]
df.dtypes

a             Int64
b           Float64
c           boolean
d    string[python]
e             Int64
f           Float64
g           boolean
h    string[python]
i    datetime64[ns]
j             Int64
dtype: object

## Specifying Categorical dtype

In [15]:
data = "col1,col2,col3\na,b,1\na,b,2\nc,d,3"
pd.read_csv(StringIO(data))

,col1,col2,col3
0,a,b,1
1,a,b,2
2,c,d,3


In [16]:
pd.read_csv(StringIO(data)).dtypes

col1    object
col2    object
col3     int64
dtype: object

In [17]:
pd.read_csv(StringIO(data), dtype="category").dtypes

col1    category
col2    category
col3    category
dtype: object

In [18]:
# Parse individual columns as Categorical
pd.read_csv(StringIO(data), dtype={"col1": "category"}).dtypes

col1    category
col2      object
col3       int64
dtype: object

In [19]:
# To control categories and data, create a CategoricalDType
from pandas.api.types import CategoricalDtype

dtype = CategoricalDtype(["d", "c", "b", "a"], ordered=True)
pd.read_csv(StringIO(data), dtype={"col1": dtype}).dtypes

col1    category
col2      object
col3       int64
dtype: object

In [20]:
# If you forget a value, it is treated as missing
dtype = CategoricalDtype(["a", "b", "d"], ordered=True)  # c is missing
pd.read_csv(StringIO(data), dtype={"col1": dtype}).col1

0      a
1      a
2    NaN
Name: col1, dtype: category
Categories (3, object): ['a' < 'b' < 'd']

In [21]:
df = pd.read_csv(StringIO(data), dtype="category")
df.dtypes

col1    category
col2    category
col3    category
dtype: object

In [22]:
df.col3

0    1
1    2
2    3
Name: col3, dtype: category
Categories (3, object): ['1', '2', '3']

In [23]:
new_categories = pd.to_numeric(df.col3.cat.categories)
df.col3 = df.col3.cat.rename_categories(new_categories)
df.col3

0    1
1    2
2    3
Name: col3, dtype: category
Categories (3, int64): [1, 2, 3]

## Naming and Using Columns

### Handling Column Names

In [24]:
# By default, pandas assumes the first row contains the column names
data = "a,b,c\n1,2,3\n4,5,6\n7,8,9"
print(data)

a,b,c
1,2,3
4,5,6
7,8,9


In [25]:
pd.read_csv(StringIO(data))

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


In [26]:
# We can specify different names to use as column names
pd.read_csv(StringIO(data), names=["foo", "bar", "baz"], header=0)

,foo,bar,baz
0,1,2,3
1,4,5,6
2,7,8,9


In [27]:
pd.read_csv(StringIO(data), names=["foo", "bar", "baz"], header=None)

,foo,bar,baz
0,a,b,c
1,1,2,3
2,4,5,6
3,7,8,9


## Duplicate Names Parsing

In [28]:
data = "a,b,a\n0,1,2\n3,4,5"
pd.read_csv(StringIO(data))

,a,b,a.1
0,0,1,2
1,3,4,5


### Filtering Columns (usecols)

In [29]:
# Select a subset of the columns using column label name OR position number
data = "a,b,c,d\n1,2,3,foo\n4,5,6,bar\n7,8,9,baz"
pd.read_csv(StringIO(data))

,a,b,c,d
0,1,2,3,foo
1,4,5,6,bar
2,7,8,9,baz


In [30]:
pd.read_csv(StringIO(data), usecols=["b", "d"])

,b,d
0,2,foo
1,5,bar
2,8,baz


In [31]:
pd.read_csv(StringIO(data), usecols=[0, 2, 3])

,a,c,d
0,1,3,foo
1,4,6,bar
2,7,9,baz


In [32]:
pd.read_csv(StringIO(data), usecols=lambda x: x.upper() in ["A", "C"])

,a,c
0,1,3
1,4,6
2,7,9


In [33]:
# Specify which columns not to use
pd.read_csv(StringIO(data), usecols=lambda x: x not in ["a", "c"])

,b,d
0,2,foo
1,5,bar
2,8,baz


## Comments and Empty Lines

### Ignoring Line Comments and Empty Lines

In [34]:
# When using the comment parameter, commented lines are ignored
data = "\na,b,c\n  \n# commented line\n1,2,3\n\n4,5,6"
print(data)


a,b,c
  
# commented line
1,2,3

4,5,6


In [35]:
pd.read_csv(StringIO(data), comment="#")

,a,b,c
0,1,2,3
1,4,5,6


In [36]:
# Can also choose to NOT ignore the blank lines
data = "a,b,c\n\n1,2,3\n\n\n4,5,6"
pd.read_csv(StringIO(data), skip_blank_lines=False)

,a,b,c
0,NaN,NaN,NaN
1,1.0,2.0,3.0
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,4.0,5.0,6.0


### Comments

In [37]:
data = (
    "ID,level,category\n"
    "Patient1,123000,x # really unpleasant\n"
    "Patient2,23000,y # wouldn't take his medicine\n"
    "Patient3,1234018,z # awesome"
)
with open("tmp.csv", "w") as fh:
    fh.write(data)

In [38]:
print(open("tmp.csv").read())

ID,level,category
Patient1,123000,x # really unpleasant
Patient2,23000,y # wouldn't take his medicine
Patient3,1234018,z # awesome


In [39]:
# This behavior is unwanted: the comments are included in the output
df = pd.read_csv("tmp.csv")
df

,ID,level,category
0,Patient1,123000,x # really unpleasant
1,Patient2,23000,y # wouldn't take his medicine
2,Patient3,1234018,z # awesome


In [40]:
# Suppress the comments
df = pd.read_csv("tmp.csv", comment="#")
df

,ID,level,category
0,Patient1,123000,x
1,Patient2,23000,y
2,Patient3,1234018,z


## Dealing with Unicode Data

In [41]:
from io import BytesIO

data = b"word,length\n" b"Tr\xc3\xa4umen,7\n" b"Gr\xc3\xbc\xc3\x9fe,5"
data = data.decode("utf8").encode("latin-1")

df = pd.read_csv(BytesIO(data), encoding="latin-1")
df

,word,length
0,Träumen,7
1,Grüße,5


## Index Columns and Trailing Delimeters

In [42]:
# If the columns are less than the values, here 3 and 4 respectively
# first column is used as the row names.
data = "a,b,c\n" "4,apple,bat,5.7\n" "8,orange,cow,10"
data

'a,b,c\n4,apple,bat,5.7\n8,orange,cow,10'

In [43]:
pd.read_csv(StringIO(data))

,a,b,c
4,apple,bat,5.7
8,orange,cow,10.0


In [44]:
pd.read_csv(StringIO(data), index_col=0)

,a,b,c
4,apple,bat,5.7
8,orange,cow,10.0


## Date Handling
### Specifying Date Columns

In [45]:
with open("foo.csv", mode="w") as f:
    f.write("date,A,B,C\n20090101,a,1,2\n20090102,b,3,4\n20090103,c,4,5")

In [46]:
# Use a column as an index, and parse it as dates.
df = pd.read_csv("foo.csv", index_col=0, parse_dates=True)
df

,A,B,C
date,,,
2009-01-01,a,1,2
2009-01-02,b,3,4
2009-01-03,c,4,5


In [47]:
# These are Python datetime objects
df.index

DatetimeIndex(['2009-01-01', '2009-01-02', '2009-01-03'], dtype='datetime64[ns]', name='date', freq=None)

In [48]:
# Sometimes, various date fiels are stored separately
# In this case, we can specify a list of column lists to be combined
data = (
    "KORD,19990127, 19:00:00, 18:56:00, 0.8100\n"
    "KORD,19990127, 20:00:00, 19:56:00, 0.0100\n"
    "KORD,19990127, 21:00:00, 20:56:00, -0.5900\n"
    "KORD,19990127, 21:00:00, 21:18:00, -0.9900\n"
    "KORD,19990127, 22:00:00, 21:56:00, -0.5900\n"
    "KORD,19990127, 23:00:00, 22:56:00, -0.5900"
)

with open("tmp.csv", "w") as fh:
    fh.write(data)

df = pd.read_csv("tmp.csv", header=None, parse_dates=[[1, 2], [1, 3]])
df

C:\Users\nkmil\AppData\Local\Temp\ipykernel_11656\1329507160.py:15: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv("tmp.csv", header=None, parse_dates=[[1, 2], [1, 3]])


,1_2,1_3,0,4
0,1999-01-27 19:00:00,1999-01-27 18:56:00,KORD,0.81
1,1999-01-27 20:00:00,1999-01-27 19:56:00,KORD,0.01
2,1999-01-27 21:00:00,1999-01-27 20:56:00,KORD,-0.59
3,1999-01-27 21:00:00,1999-01-27 21:18:00,KORD,-0.99
4,1999-01-27 22:00:00,1999-01-27 21:56:00,KORD,-0.59
5,1999-01-27 23:00:00,1999-01-27 22:56:00,KORD,-0.59


In [49]:
# Use a dict for custom name columns
date_spec = {"nominal": [1, 2], "actual": [1, 3]}

df = pd.read_csv("tmp.csv", header=None, parse_dates=date_spec)
df

C:\Users\nkmil\AppData\Local\Temp\ipykernel_11656\1298113533.py:4: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv("tmp.csv", header=None, parse_dates=date_spec)


,nominal,actual,0,4
0,1999-01-27 19:00:00,1999-01-27 18:56:00,KORD,0.81
1,1999-01-27 20:00:00,1999-01-27 19:56:00,KORD,0.01
2,1999-01-27 21:00:00,1999-01-27 20:56:00,KORD,-0.59
3,1999-01-27 21:00:00,1999-01-27 21:18:00,KORD,-0.99
4,1999-01-27 22:00:00,1999-01-27 21:56:00,KORD,-0.59
5,1999-01-27 23:00:00,1999-01-27 22:56:00,KORD,-0.59


### Date Parsing Functions - Custom date_format

In [50]:
date_spec = {"nominal": [1, 2], "actual": [1, 3]}
date_format = {"nominal": "%d%m%Y"}

df = pd.read_csv("tmp.csv", header=None, parse_dates=date_spec, date_format=date_format)
df

C:\Users\nkmil\AppData\Local\Temp\ipykernel_11656\360829160.py:4: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv("tmp.csv", header=None, parse_dates=date_spec, date_format=date_format)


,nominal,actual,0,4
0,19990127 19:00:00,1999-01-27 18:56:00,KORD,0.81
1,19990127 20:00:00,1999-01-27 19:56:00,KORD,0.01
2,19990127 21:00:00,1999-01-27 20:56:00,KORD,-0.59
3,19990127 21:00:00,1999-01-27 21:18:00,KORD,-0.99
4,19990127 22:00:00,1999-01-27 21:56:00,KORD,-0.59
5,19990127 23:00:00,1999-01-27 22:56:00,KORD,-0.59


### Parsing a CSV with Mixed Timezones

In [51]:
content = """\
a
2000-01-01T00:00:00+05:00
2000-01-01T00:00:00+06:00
"""

df = pd.read_csv(StringIO(content))
df["a"] = pd.to_datetime(df["a"], utc=True)

df["a"]

0   1999-12-31 19:00:00+00:00
1   1999-12-31 18:00:00+00:00
Name: a, dtype: datetime64[ns, UTC]

### Inferring datetime Format

In [52]:
df = pd.read_csv("foo.csv", index_col=0, parse_dates=True)
df

,A,B,C
date,,,
2009-01-01,a,1,2
2009-01-02,b,3,4
2009-01-03,c,4,5


In [53]:
# For mixed datetime formats:
data = StringIO("date\n12 Jan 2000\n2000-01-13\n")
df = pd.read_csv(data)
df.date = pd.to_datetime(df.date, format="mixed")
df

,date
0,2000-01-12
1,2000-01-13


### International Date Formats

In [54]:
# Specify that date comes first (and not the month)
data = "date,value,cat\n1/6/2000,5,a\n2/6/2000,10,b\n3/6/2000,15,c"
print(data)

date,value,cat
1/6/2000,5,a
2/6/2000,10,b
3/6/2000,15,c


In [55]:
with open("tmp.csv", "w") as fh:
    fh.write(data)

pd.read_csv("tmp.csv", parse_dates=[0])

,date,value,cat
0,2000-01-06,5,a
1,2000-02-06,10,b
2,2000-03-06,15,c


In [56]:
pd.read_csv("tmp.csv", parse_dates=[0], dayfirst=True, date_format="%dd%mm%Y")

,date,value,cat
0,1/6/2000,5,a
1,2/6/2000,10,b
2,3/6/2000,15,c


### Writing CSVs to Binary File Objects

In [57]:
import io

data = pd.DataFrame([0, 1, 2])
buffer = io.BytesIO()
data.to_csv(buffer, encoding="utf-8", compression="gzip")

## Specifying Methods for Floating-Point Conversion

In [58]:
val = "0.3066101993807095471566981359501369297504425048828125"
data = "a,b,c\n1,2,{0}".format(val)
abs(pd.read_csv(StringIO(data), engine="c", float_precision=None)["c"][0] - float(val))

np.float64(5.551115123125783e-17)

In [59]:
abs(pd.read_csv(StringIO(data), engine="c", float_precision="high")["c"][0] - float(val))

np.float64(5.551115123125783e-17)

In [60]:
abs(pd.read_csv(StringIO(data), engine="c", float_precision="round_trip")["c"][0] - float(val))

np.float64(0.0)

## Thousand Separators

In [61]:
data = "ID|level|category\n" "Patient1|123,000|x\n" "Patient2|23,000|y\n" "Patient3|1,234,018|z"

with open("tmp.csv", "w") as fh:
    fh.write(data)

df = pd.read_csv("tmp.csv", sep="|")
df

,ID,level,category
0,Patient1,"123,000",x
1,Patient2,"23,000",y
2,Patient3,"1,234,018",z


In [62]:
# NOTE: by default numbers with a thousand separator, are parsed as strings
df.level.dtype

dtype('O')

In [63]:
# Specify the correct thousand separator, so that the values are parsed correctly
df = pd.read_csv("tmp.csv", sep="|", thousands=",")
df

,ID,level,category
0,Patient1,123000,x
1,Patient2,23000,y
2,Patient3,1234018,z


In [64]:
# Now level column values are correctly parsed as integers
df.level.dtype

dtype('int64')

## NA Values

In [65]:
# Use na_values and keep_default_na
# na_values=["NA", "not"]
# keep_default_na = True : recognize NaN as a missing value

## Boolean Values

In [66]:
data = "a,b,c\n1,Yes,2\n3,No,4"
print(data)

a,b,c
1,Yes,2
3,No,4


In [67]:
pd.read_csv(StringIO(data))

,a,b,c
0,1,Yes,2
1,3,No,4


In [68]:
pd.read_csv(StringIO(data), true_values=["Yes"], false_values=["No"])

,a,b,c
0,1,True,2
1,3,False,4


## Handling "Bad" Lines

In [69]:
# A line with too few fields, will result in NA values filled in
# A line with too many fields, will raise an error

# data = "a,b,c\n1,2,3\n4,5,6,7\n8,9,10"  # the third line has four values instead of expected three
# pd.read_csv(StringIO(data))

In [70]:
# One solution is to skip bad lines
pd.read_csv(StringIO(data), on_bad_lines="skip")

,a,b,c
0,1,Yes,2
1,3,No,4


In [71]:
# Can also pass a callable function to handle bad lines
external_list = []


def bad_lines_func(line):
    external_list.append(line)
    return line[-3:]


external_list

[]

In [72]:
bad_lines_func = lambda line: print(line)
data = "name,type\nname a,a is of type a\nname b,'b' is of type b'"
data

"name,type\nname a,a is of type a\nname b,'b' is of type b'"

In [73]:
pd.read_csv(StringIO(data), on_bad_lines=bad_lines_func, engine="python")

,name,type
0,name a,a is of type a
1,name b,'b' is of type b'


## Dialect

In [74]:
data = "label1,label2,label3\n" 'index1,"a,c,e\n' "index2,b,d,f"
print(data)

label1,label2,label3
index1,"a,c,e
index2,b,d,f


In [75]:
# By default, read_csv uses the Excle dialect, meaning if it finds a single double quote before
# the new line character, it will fail.
import csv

dia = csv.excel()
dia.quoting = csv.QUOTE_NONE
pd.read_csv(StringIO(data), dialect=dia)

,label1,label2,label3
index1,"""a",c,e
index2,b,d,f


In [76]:
# dialect opitions can also be specified with keywords
data = "a,b,c~1,2,3~4,5,6"
pd.read_csv(StringIO(data), lineterminator="~")

,a,b,c
0,1,2,3
1,4,5,6


In [77]:
# To skip any whitespace after a delimeter
data = "a, b, c\n1, 2, 3\n4, 5 ,6"
print(data)

a, b, c
1, 2, 3
4, 5 ,6


In [78]:
pd.read_csv(StringIO(data), skipinitialspace=True)

,a,b,c
0,1,2,3
1,4,5,6


## Quoting & Escape Characters

In [79]:
data = 'a,b\n"hello, \\"Bob\\", nice to see you",5'
print(data)

a,b
"hello, \"Bob\", nice to see you",5


In [80]:
pd.read_csv(StringIO(data), escapechar="\\")

,a,b
0,"hello, ""Bob"", nice to see you",5


## Files with Fixed Width Columns

In [81]:
data1 = (
    "id8141    360.242940   149.910199   11950.7\n"
    "id1594    444.953632   166.985655   11788.4\n"
    "id1849    364.136849   183.628767   11806.2\n"
    "id1230    413.836124   184.375703   11916.8\n"
    "id1948    502.953953   173.237159   12468.3"
)

with open("bar.csv", "w") as f:
    f.write(data1)

# Define the extent of the fixed-width fields of each line as half-open intervals
colspecs = [(0, 6), (8, 20), (21, 33), (34, 43)]
df = pd.read_fwf("bar.csv", colspecs=colspecs, header=None, index_col=0)
df

,1,2,3
0,,,
id8141,360.242940,149.910199,11950.7
id1594,444.953632,166.985655,11788.4
id1849,364.136849,183.628767,11806.2
id1230,413.836124,184.375703,11916.8
id1948,502.953953,173.237159,12468.3


In [82]:
# Or specify the column widths
widths = [6, 14, 13, 10]
df = pd.read_fwf("bar.csv", widths=widths, header=None)
df

,0,1,2,3
0,id8141,360.242940,149.910199,11950.7
1,id1594,444.953632,166.985655,11788.4
2,id1849,364.136849,183.628767,11806.2
3,id1230,413.836124,184.375703,11916.8
4,id1948,502.953953,173.237159,12468.3


## Indexes
### Files with an "implicit" index column

In [83]:
# This file is missing a header label (4 values on each row, but only 3 column labels)
# So it assumes first column is the index.
data = "A,B,C\n20090101,a,1,2\n20090102,b,3,4\n20090103,c,4,5"
print(data)

A,B,C
20090101,a,1,2
20090102,b,3,4
20090103,c,4,5


In [84]:
with open("foo.csv", "w") as f:
    f.write(data)

In [85]:
pd.read_csv("foo.csv")

,A,B,C
20090101,a,1,2
20090102,b,3,4
20090103,c,4,5


In [86]:
# To parse the dates:
df = pd.read_csv("foo.csv", parse_dates=True)
df.index

DatetimeIndex(['2009-01-01', '2009-01-02', '2009-01-03'], dtype='datetime64[ns]', freq=None)

### Reading an index with a MultiIndex

In [87]:
# Here the data are indexed by two columns
data = 'year,indiv,zit,xit\n1977,"A",1.2,.6\n1977,"B",1.5,5'
print(data)

year,indiv,zit,xit
1977,"A",1.2,.6
1977,"B",1.5,5


In [88]:
with open("mindex_ex.csv", mode="w") as f:
    f.write(data)

In [89]:
df = pd.read_csv("mindex_ex.csv", index_col=[0, 1])
df

zit  xit
year indiv          
1977 A      1.2  0.6
     B      1.5  5.0

In [90]:
df.loc[1977]

,zit,xit
indiv,,
A,1.2,0.6
B,1.5,5.0


### Reading columns with a MultiIndex

In [91]:
mi_idx = pd.MultiIndex.from_arrays([[1, 2, 3, 4], list("abcd")], names=list("ab"))
mi_col = pd.MultiIndex.from_arrays([[1, 2], list("ab")], names=list("cd"))
df = pd.DataFrame(np.ones((4, 2)), index=mi_idx, columns=mi_col)
df.to_csv("mi.csv")

In [92]:
print(open("mi.csv").read())

c,,1,2
d,,a,b
a,b,,
1,a,1.0,1.0
2,b,1.0,1.0
3,c,1.0,1.0
4,d,1.0,1.0



In [93]:
pd.read_csv("mi.csv", header=[0, 1, 2, 3], index_col=[0, 1])

,c,1,2
,d,a,b
,a,Unnamed: 2_level_2,Unnamed: 3_level_2
,1,1.0,1.0
2,b,1.0,1.0
3,c,1.0,1.0
4,d,1.0,1.0


## Automatically "sniffing" the delimiter

In [94]:
df = pd.DataFrame(np.random.randn(10, 4))
df.to_csv("tmp2.csv", sep=":", index=False)

# To let pandas automatically "sniff" the delimiter, use sep=None
pd.read_csv("tmp2.csv", sep=None, engine="python")

,0,1,2,3
0,-1.392942,-0.423959,-0.104298,-0.704415
1,2.378452,-1.212562,0.065868,-1.870270
2,-1.428582,0.062031,-0.951599,-0.413196
3,-0.018274,-0.415483,-0.106588,1.750841
4,1.093548,1.155788,0.272478,-1.134052
5,1.008403,0.675537,-0.710397,1.055526
6,2.517225,0.880436,-0.809485,-1.749746
7,-1.589895,0.679340,-1.221101,-0.080867
8,0.196591,0.256436,1.683765,-1.652041
9,0.842928,0.294260,0.348779,-1.785186


## Iterating through files chunk by chunk

In [95]:
# Instead of reading the entire file into memory, we can iterate lazily
df = pd.DataFrame(np.random.randn(10, 4))
df.to_csv("tmp.csv", index=False)
table = pd.read_csv("tmp.csv")
table

,0,1,2,3
0,1.250803,-0.429414,-0.791839,-0.241301
1,1.005242,0.036875,-1.594414,-0.943849
2,-2.357529,1.162605,-0.106901,0.482285
3,0.482157,-0.290534,-0.210407,1.051952
4,0.097623,-0.298432,0.501052,0.492821
5,0.365902,0.525289,-0.547724,-0.724962
6,1.009222,-0.664854,2.156469,0.803095
7,0.639303,0.300728,0.563393,-1.098567
8,-0.320376,-1.209049,-0.507031,-0.290685
9,-1.302773,-0.920508,-0.687983,1.636919


In [96]:
# Now iterate using chunksize which returns a TextFileReader:
with pd.read_csv("tmp.csv", chunksize=4) as reader:
    print(reader)
    for chunk in reader:
        print(chunk)

          0         1         2         3
0  1.250803 -0.429414 -0.791839 -0.241301
1  1.005242  0.036875 -1.594414 -0.943849
2 -2.357529  1.162605 -0.106901  0.482285
3  0.482157 -0.290534 -0.210407  1.051952
          0         1         2         3
4  0.097623 -0.298432  0.501052  0.492821
5  0.365902  0.525289 -0.547724 -0.724962
6  1.009222 -0.664854  2.156469  0.803095
7  0.639303  0.300728  0.563393 -1.098567
          0         1         2         3
8 -0.320376 -1.209049 -0.507031 -0.290685
9 -1.302773 -0.920508 -0.687983  1.636919


## Specifying the parser engine

## Reading/writing remote files

# JSON

In [97]:
dfj = pd.DataFrame(np.random.randn(5, 2), columns=list("AB"))
json = dfj.to_json()
json

'{"A":{"0":-1.6303084204,"1":-0.0259133873,"2":0.3163161072,"3":0.61683461,"4":-0.2186424852},"B":{"0":0.060753692,"1":-0.2292515781,"2":-0.6183719471,"3":0.3028567292,"4":1.9540723634}}'

## Orient options

In [98]:
dfjo = pd.DataFrame(dict(A=range(1, 4), B=range(4, 7), C=range(7, 10)), columns=list("ABC"), index=list("xyz"))
dfjo

,A,B,C
x,1,4,7
y,2,5,8
z,3,6,9


In [99]:
sjo = pd.Series(dict(x=15, y=16, z=17), name="D")
sjo

x    15
y    16
z    17
Name: D, dtype: int64

In [100]:
# For DataFrames the default is column-oriented: column labels act as the primary index
dfjo.to_json(orient="columns")

'{"A":{"x":1,"y":2,"z":3},"B":{"x":4,"y":5,"z":6},"C":{"x":7,"y":8,"z":9}}'

In [101]:
# For Series the default is index-oriented: index labels act as the primary index
dfjo.to_json(orient="index")

'{"x":{"A":1,"B":4,"C":7},"y":{"A":2,"B":5,"C":8},"z":{"A":3,"B":6,"C":9}}'

In [102]:
sjo.to_json(orient="index")

'{"x":15,"y":16,"z":17}'

In [103]:
# Record-oriented; index labels are not included
dfj.to_json(orient="records")

'[{"A":-1.6303084204,"B":0.060753692},{"A":-0.0259133873,"B":-0.2292515781},{"A":0.3163161072,"B":-0.6183719471},{"A":0.61683461,"B":0.3028567292},{"A":-0.2186424852,"B":1.9540723634}]'

In [104]:
# Value-oriented: column and index labels not included
dfjo.to_json(orient="values")

'[[1,4,7],[2,5,8],[3,6,9]]'

In [105]:
# Split-oriented: separate entries for data, index and columns
dfjo.to_json(orient="split")

'{"columns":["A","B","C"],"index":["x","y","z"],"data":[[1,4,7],[2,5,8],[3,6,9]]}'

## Date handling

In [106]:
# ISO Date Format
dfd = pd.DataFrame(np.random.randn(5, 2), columns=list("AB"))

In [107]:
dfd["date"] = pd.Timestamp("20130101")

In [108]:
# Sorts labels along horizontal axis (columns) DESC i.e. B comes before A
dfd = dfd.sort_index(axis=1, ascending=False)
dfd

,date,B,A
0,2013-01-01,0.680826,-0.267407
1,2013-01-01,1.240254,-0.173953
2,2013-01-01,-0.583456,1.401932
3,2013-01-01,0.237865,1.361604
4,2013-01-01,-0.660646,-0.017670


In [109]:
json = dfd.to_json(date_format="iso")

In [110]:
json

'{"date":{"0":"2013-01-01T00:00:00.000","1":"2013-01-01T00:00:00.000","2":"2013-01-01T00:00:00.000","3":"2013-01-01T00:00:00.000","4":"2013-01-01T00:00:00.000"},"B":{"0":0.6808260238,"1":1.240254096,"2":-0.5834557833,"3":0.2378650732,"4":-0.6606458449},"A":{"0":-0.2674074073,"1":-0.1739532469,"2":1.4019316498,"3":1.3616044671,"4":-0.0176698649}}'

## Fallback Behavior

## Reading JSON

### Data conversion

In [111]:
from io import StringIO

pd.read_json(StringIO(json))

,date,B,A
0,2013-01-01,0.680826,-0.267407
1,2013-01-01,1.240254,-0.173953
2,2013-01-01,-0.583456,1.401932
3,2013-01-01,0.237865,1.361604
4,2013-01-01,-0.660646,-0.017670


In [112]:
# Writing to a file with a date index and a date column
dfj2 = dfj.copy()
dfj2["date"] = pd.Timestamp("20130101")
dfj2["ints"] = list(range(5))
dfj2["bools"] = True
dfj2.index = pd.date_range("20130101", periods=5)
dfj2.to_json("test.json")
with open("test.json") as fh:
    print(fh.read())

{"A":{"1356998400000":-1.6303084204,"1357084800000":-0.0259133873,"1357171200000":0.3163161072,"1357257600000":0.61683461,"1357344000000":-0.2186424852},"B":{"1356998400000":0.060753692,"1357084800000":-0.2292515781,"1357171200000":-0.6183719471,"1357257600000":0.3028567292,"1357344000000":1.9540723634},"date":{"1356998400000":1356,"1357084800000":1356,"1357171200000":1356,"1357257600000":1356,"1357344000000":1356},"ints":{"1356998400000":0,"1357084800000":1,"1357171200000":2,"1357257600000":3,"1357344000000":4},"bools":{"1356998400000":true,"1357084800000":true,"1357171200000":true,"1357257600000":true,"1357344000000":true}}


In [113]:
# Reading from a file
pd.read_json("test.json")

,A,B,date,ints,bools
2013-01-01,-1.630308,0.060754,1356,0,True
2013-01-02,-0.025913,-0.229252,1356,1,True
2013-01-03,0.316316,-0.618372,1356,2,True
2013-01-04,0.616835,0.302857,1356,3,True
2013-01-05,-0.218642,1.954072,1356,4,True


In [114]:
pd.read_json("test.json", dtype=object).dtypes

A        object
B        object
date     object
ints     object
bools    object
dtype: object

In [115]:
# Specify dtypes for conversion
pd.read_json("test.json", dtype={"A": "float32", "bools": "int8"}).dtypes

A        float32
B        float64
date       int64
ints       int64
bools       int8
dtype: object

In [116]:
# Preserve string indices
from io import StringIO

si = pd.DataFrame(np.zeros((4, 4)), columns=list(range(4)), index=[str(i) for i in range(4)])
si

,0,1,2,3
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0


In [117]:
si.index

Index(['0', '1', '2', '3'], dtype='object')

In [118]:
si.columns

Index([0, 1, 2, 3], dtype='int64')

In [119]:
json = si.to_json()
sij = pd.read_json(StringIO(json), convert_axes=False)
sij

,0,1,2,3
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0


In [120]:
sij.index

Index(['0', '1', '2', '3'], dtype='object')

In [121]:
sij.columns

Index(['0', '1', '2', '3'], dtype='object')

In [122]:
# Dates written in nanoseconds, need to be read back in nanoseconds
json = dfj2.to_json(date_unit="ns")

# Try to parse timestamps as milliseconds -> Won't work!
dfju = pd.read_json(StringIO(json), date_unit="ms")
dfju

,A,B,date,ints,bools
1356998400000000000,-1.630308,0.060754,1356998400,0,True
1357084800000000000,-0.025913,-0.229252,1356998400,1,True
1357171200000000000,0.316316,-0.618372,1356998400,2,True
1357257600000000000,0.616835,0.302857,1356998400,3,True
1357344000000000000,-0.218642,1.954072,1356998400,4,True


In [123]:
# Let pandas detect the correct precision
dfju = pd.read_json(StringIO(json))
dfju

,A,B,date,ints,bools
2013-01-01,-1.630308,0.060754,2013-01-01,0,True
2013-01-02,-0.025913,-0.229252,2013-01-01,1,True
2013-01-03,0.316316,-0.618372,2013-01-01,2,True
2013-01-04,0.616835,0.302857,2013-01-01,3,True
2013-01-05,-0.218642,1.954072,2013-01-01,4,True


In [124]:
data = (
    '{"a":{"0":1,"1":3},"b":{"0":2.5,"1":4.5},"c":{"0":true,"1":false},"d":{"0":"a","1":"b"},'
    '"e":{"0":null,"1":6.0},"f":{"0":null,"1":7.5},"g":{"0":null,"1":true},"h":{"0":null,"1":"a"},'
    '"i":{"0":"12-31-2019","1":"12-31-2019"},"j":{"0":null,"1":null}}'
)

df = pd.read_json(StringIO(data), dtype_backend="pyarrow")
df

,a,b,c,d,e,f,g,h,i,j
0,1,2.5,True,a,<NA>,<NA>,<NA>,<NA>,12-31-2019,None
1,3,4.5,False,b,6,7.5,True,a,12-31-2019,None


## Normalization

In [125]:
data = [
    {"id": 1, "name": {"first": "Coleen", "last": "Volk"}},
    {"name": {"given": "Mark", "family": "Regner"}},
    {"id": 2, "name": "Faye Raker"},
]

pd.json_normalize(data)

,id,name.first,name.last,name.given,name.family,name
0,1.0,Coleen,Volk,NaN,NaN,NaN
1,NaN,NaN,NaN,Mark,Regner,NaN
2,2.0,NaN,NaN,NaN,NaN,Faye Raker


In [126]:
data = [
    {
        "state": "Florida",
        "shortname": "FL",
        "info": {"governor": "Rick Scott"},
        "county": [
            {"name": "Dade", "population": 12345},
            {"name": "Broward", "population": 40000},
            {"name": "Palm Beach", "population": 60000},
        ],
    },
    {
        "state": "Ohio",
        "shortname": "OH",
        "info": {"governor": "John Kasich"},
        "county": [
            {"name": "Summit", "population": 1234},
            {"name": "Cuyahoga", "population": 1337},
        ],
    },
]

pd.json_normalize(data, "county", ["state", "shortname", ["info", "governor"]])

,name,population,state,shortname,info.governor
0,Dade,12345,Florida,FL,Rick Scott
1,Broward,40000,Florida,FL,Rick Scott
2,Palm Beach,60000,Florida,FL,Rick Scott
3,Summit,1234,Ohio,OH,John Kasich
4,Cuyahoga,1337,Ohio,OH,John Kasich


In [127]:
data = [
    {
        "CreatedBy": {"Name": "User001"},
        "Lookup": {
            "TextField": "Some text",
            "UserField": {"Id": "ID001", "Name": "Name001"},
        },
        "Image": {"a": "b"},
    }
]

pd.json_normalize(data, max_level=1)

,CreatedBy.Name,Lookup.TextField,Lookup.UserField,Image.a
0,User001,Some text,"{'Id': 'ID001', 'Name': 'Name001'}",b


## Line delimited json

In [128]:
jsonl = """
    {"a": 1, "b": 2}
    {"a": 3, "b": 4}
"""
df = pd.read_json(StringIO(jsonl), lines=True)
df

,a,b
0,1,2
1,3,4


In [129]:
df.to_json(orient="records", lines=True)

'{"a":1,"b":2}\n{"a":3,"b":4}\n'

In [130]:
# reader is an iterator that returns "chunksize" lines each iteration
with pd.read_json(StringIO(jsonl), lines=True, chunksize=1) as reader:
    reader
    for chunk in reader:
        print(chunk)

Empty DataFrame
Columns: []
Index: []
   a  b
0  1  2
   a  b
1  3  4


In [131]:
from io import BytesIO

df = pd.read_json(BytesIO(jsonl.encode()), lines=True, engine="pyarrow")
df

,a,b
0,1,2
1,3,4


## Table Schema

In [132]:
df = pd.DataFrame(
    {
        "A": [1, 2, 3],
        "B": ["a", "b", "c"],
        "C": pd.date_range("2016-01-01", freq="d", periods=3),
    },
    index=pd.Index(range(3), name="idx"),
)

df.to_json(orient="table", date_format="iso")

'{"schema":{"fields":[{"name":"idx","type":"integer"},{"name":"A","type":"integer"},{"name":"B","type":"string"},{"name":"C","type":"datetime"}],"primaryKey":["idx"],"pandas_version":"1.4.0"},"data":[{"idx":0,"A":1,"B":"a","C":"2016-01-01T00:00:00.000"},{"idx":1,"A":2,"B":"b","C":"2016-01-02T00:00:00.000"},{"idx":2,"A":3,"B":"c","C":"2016-01-03T00:00:00.000"}]}'

In [133]:
from pandas.io.json import build_table_schema

s = pd.Series(pd.date_range("2016", periods=4))
build_table_schema(s)

{'fields': [{'name': 'index', 'type': 'integer'},
  {'name': 'values', 'type': 'datetime'}],
 'primaryKey': ['index'],
 'pandas_version': '1.4.0'}

In [134]:
s_tz = pd.Series(pd.date_range("2016", periods=12, tz="US/Central"))
build_table_schema(s_tz)

{'fields': [{'name': 'index', 'type': 'integer'},
  {'name': 'values', 'type': 'datetime', 'tz': 'US/Central'}],
 'primaryKey': ['index'],
 'pandas_version': '1.4.0'}

In [135]:
s_per = pd.Series(1, index=pd.period_range("2016", freq="Y-DEC", periods=4))
build_table_schema(s_per)

{'fields': [{'name': 'index', 'type': 'datetime', 'freq': 'YE-DEC'},
  {'name': 'values', 'type': 'integer'}],
 'primaryKey': ['index'],
 'pandas_version': '1.4.0'}

In [136]:
s_cat = pd.Series(pd.Categorical(["a", "b", "a"]))
build_table_schema(s_cat)

{'fields': [{'name': 'index', 'type': 'integer'},
  {'name': 'values',
   'type': 'any',
   'constraints': {'enum': ['a', 'b']},
   'ordered': False}],
 'primaryKey': ['index'],
 'pandas_version': '1.4.0'}

In [137]:
s_dupe = pd.Series([1, 2], index=[1, 1])
build_table_schema(s_dupe)

{'fields': [{'name': 'index', 'type': 'integer'},
  {'name': 'values', 'type': 'integer'}],
 'pandas_version': '1.4.0'}

In [138]:
s_multi = pd.Series(1, index=pd.MultiIndex.from_product([("a", "b"), (0, 1)]))
build_table_schema(s_multi)

{'fields': [{'name': 'level_0', 'type': 'string'},
  {'name': 'level_1', 'type': 'integer'},
  {'name': 'values', 'type': 'integer'}],
 'primaryKey': FrozenList(['level_0', 'level_1']),
 'pandas_version': '1.4.0'}

In [139]:
df = pd.DataFrame(
    {
        "foo": [1, 2, 3, 4],
        "bar": ["a", "b", "c", "d"],
        "baz": pd.date_range("2018-01-01", freq="d", periods=4),
        "qux": pd.Categorical(["a", "b", "c", "c"]),
    },
    index=pd.Index(range(4), name="idx"),
)

df

,foo,bar,baz,qux
idx,,,,
0,1,a,2018-01-01,a
1,2,b,2018-01-02,b
2,3,c,2018-01-03,c
3,4,d,2018-01-04,c


In [140]:
df.dtypes

foo             int64
bar            object
baz    datetime64[ns]
qux          category
dtype: object

In [141]:
df.to_json("test.json", orient="table")
new_df = pd.read_json("test.json", orient="table")
new_df

,foo,bar,baz,qux
idx,,,,
0,1,a,2018-01-01,a
1,2,b,2018-01-02,b
2,3,c,2018-01-03,c
3,4,d,2018-01-04,c


In [142]:
new_df.dtypes

foo             int64
bar            object
baz    datetime64[ns]
qux          category
dtype: object

# HTML

In [143]:
url = "https://www.fdic.gov/resources/resolutions/bank-failures/failed-bank-list"
pd.read_html(url)

[                                            Bank Name                City  \
 0                        The Santa Anna National Bank          Santa Anna   
 1                                Pulaski Savings Bank             Chicago   
 2                  The First National Bank of Lindsay             Lindsay   
 3               Republic First Bank dba Republic Bank        Philadelphia   
 4                                       Citizens Bank            Sac City   
 5                            Heartland Tri-State Bank             Elkhart   
 6                                 First Republic Bank       San Francisco   
 7                                      Signature Bank            New York   
 8                                 Silicon Valley Bank         Santa Clara   
 9                                   Almena State Bank              Almena   
 10                         First City Bank of Florida   Fort Walton Beach   
 11                               The First State Bank       Bar

In [144]:
html_str = """
            <table>
                <tr>
                    <th>A</th>
                    <th colspan="1">B</th>
                    <th rowspan="1">C</th>
                </tr>
                <tr>
                    <td>a</td>
                    <td>b</td>
                    <td>c</td>
                </tr>
            </table>
"""

with open("tmp.html", "w") as f:
    f.write(html_str)

# read_html returns a list of DataFrames
df = pd.read_html("tmp.html")

In [145]:
df[0]

,A,B,C
0,a,b,c


In [146]:
dfs = pd.read_html(StringIO(html_str))
df[0]

,A,B,C
0,a,b,c


In [147]:
# Extract links from cells
html_table = """
<table>
  <tr>
    <th>GitHub</th>
  </tr>
  <tr>
    <td><a href="https://github.com/pandas-dev/pandas">pandas</a></td>
  </tr>
</table>
"""

df = pd.read_html(StringIO(html_table), extract_links="all")[0]
df

,"(GitHub, None)"
0,"(pandas, https://github.com/pandas-dev/pandas)"


In [148]:
df[("GitHub", None)]

0    (pandas, https://github.com/pandas-dev/pandas)
Name: (GitHub, None), dtype: object

In [149]:
df[("GitHub", None)].str[1]

0    https://github.com/pandas-dev/pandas
Name: (GitHub, None), dtype: object

## Writing to HTML Files

In [150]:
from IPython.display import HTML, display

df = pd.DataFrame(np.random.randn(2, 2))
df

,0,1
0,1.170578,-1.003585
1,1.439389,-0.342078


In [151]:
html = df.to_html()
print(html)  # the raw HTML

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>0</th>
      <th>1</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1.170578</td>
      <td>-1.003585</td>
    </tr>
    <tr>
      <th>1</th>
      <td>1.439389</td>
      <td>-0.342078</td>
    </tr>
  </tbody>
</table>


In [152]:
display(HTML(html))  # render the HTML in the notebook

,0,1
0,1.170578,-1.003585
1,1.439389,-0.342078


In [153]:
# We can limit the columns shown
html = df.to_html(columns=[0])
print(html)  # the raw HTML

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>0</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1.170578</td>
    </tr>
    <tr>
      <th>1</th>
      <td>1.439389</td>
    </tr>
  </tbody>
</table>


In [154]:
display(HTML(html))

,0
0,1.170578
1,1.439389


In [155]:
# Control precision of the floating point numbers
html = df.to_html(float_format="%.10f")
print(html)

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>0</th>
      <th>1</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1.1705783929</td>
      <td>-1.0035854652</td>
    </tr>
    <tr>
      <th>1</th>
      <td>1.4393885987</td>
      <td>-0.3420781895</td>
    </tr>
  </tbody>
</table>


In [156]:
display(HTML(html))  # render the HTML in the notebook

,0,1
0,1.1705783929,-1.0035854652
1,1.4393885987,-0.3420781895


In [157]:
# Make the row labels bold by default
html = df.to_html(bold_rows=False)
print(html)

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>0</th>
      <th>1</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>1.170578</td>
      <td>-1.003585</td>
    </tr>
    <tr>
      <td>1</td>
      <td>1.439389</td>
      <td>-0.342078</td>
    </tr>
  </tbody>
</table>


In [158]:
display(HTML(html))

,0,1
0,1.170578,-1.003585
1,1.439389,-0.342078


In [159]:
# Give the resulting HTML table, CSS classes

In [160]:
# Add hyperlinks to cells containing URLs
url_df = pd.DataFrame(
    {
        "name": ["Python", "pandas"],
        "url": ["https://www.python.org", "https://pandas.pydata.org"],
    }
)

html = url_df.to_html(render_links=True)
print(html)

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>name</th>
      <th>url</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Python</td>
      <td><a href="https://www.python.org" target="_blank">https://www.python.org</a></td>
    </tr>
    <tr>
      <th>1</th>
      <td>pandas</td>
      <td><a href="https://pandas.pydata.org" target="_blank">https://pandas.pydata.org</a></td>
    </tr>
  </tbody>
</table>


In [161]:
display(HTML(html))  # render the HTML in the notebook

,name,url
0,Python,https://www.python.org
1,pandas,https://pandas.pydata.org


In [162]:
df = pd.DataFrame({"a": list("&<>"), "b": np.random.randn(3)})

In [163]:
# By default, special characters are escaped
html = df.to_html()
print(html)

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>a</th>
      <th>b</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>&amp;</td>
      <td>0.156804</td>
    </tr>
    <tr>
      <th>1</th>
      <td>&lt;</td>
      <td>-2.154975</td>
    </tr>
    <tr>
      <th>2</th>
      <td>&gt;</td>
      <td>-0.564525</td>
    </tr>
  </tbody>
</table>


In [164]:
display(HTML(html))  # render the HTML in the notebook

,a,b
0,&,0.156804
1,<,-2.154975
2,>,-0.564525


In [165]:
# Not ecaped
html = df.to_html(escape=False)
print(html)

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>a</th>
      <th>b</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>&</td>
      <td>0.156804</td>
    </tr>
    <tr>
      <th>1</th>
      <td><</td>
      <td>-2.154975</td>
    </tr>
    <tr>
      <th>2</th>
      <td>></td>
      <td>-0.564525</td>
    </tr>
  </tbody>
</table>


In [166]:
display(HTML(html))  # render the HTML in the notebook

,a,b
0,&,0.156804
1,<,-2.154975
2,>,-0.564525


## HTML Table Parsing Gotchas

# LaTeX

## Writing to LaTeX Files

In [167]:
df = pd.DataFrame([[1, 2], [3, 4]], index=["a", "b"], columns=["c", "d"])
print(df)
print(df.style.to_latex())

   c  d
a  1  2
b  3  4
\begin{tabular}{lrr}
 & c & d \\
a & 1 & 2 \\
b & 3 & 4 \\
\end{tabular}



# XML
## Reading XML

In [168]:
# Read an XML string
xml = """<?xml version="1.0" encoding="UTF-8"?>
<bookstore>
  <book category="cooking">
    <title lang="en">Everyday Italian</title>
    <author>Giada De Laurentiis</author>
    <year>2005</year>
    <price>30.00</price>
  </book>
  <book category="children">
    <title lang="en">Harry Potter</title>
    <author>J K. Rowling</author>
    <year>2005</year>
    <price>29.99</price>
  </book>
  <book category="web">
    <title lang="en">Learning XML</title>
    <author>Erik T. Ray</author>
    <year>2003</year>
    <price>39.95</price>
  </book>
</bookstore>"""

In [169]:
df = pd.read_xml(StringIO(xml))
df

,category,title,author,year,price
0,cooking,Everyday Italian,Giada De Laurentiis,2005,30.00
1,children,Harry Potter,J K. Rowling,2005,29.99
2,web,Learning XML,Erik T. Ray,2003,39.95


In [170]:
# Read a URL with no options
df = pd.read_xml("https://www.w3schools.com/xml/books.xml")
df

,category,title,author,year,price,cover
0,cooking,Everyday Italian,Giada De Laurentiis,2005,30.00,None
1,children,Harry Potter,J K. Rowling,2005,29.99,None
2,web,XQuery Kick Start,Vaidyanathan Nagarajan,2003,49.99,None
3,web,Learning XML,Erik T. Ray,2003,39.95,paperback


In [171]:
# Read the contents of teh file, and pass it to read_xml as a string
file_path = "books.xml"
with open(file_path, "w") as f:
    f.write(xml)

In [172]:
with open(file_path, "r") as f:
    df = pd.read_xml(StringIO(f.read()))

df

,category,title,author,year,price
0,cooking,Everyday Italian,Giada De Laurentiis,2005,30.00
1,children,Harry Potter,J K. Rowling,2005,29.99
2,web,Learning XML,Erik T. Ray,2003,39.95


# Excel Files
## Reading Excel Files

# HDF5 (PyTables)
<b>HDFStore</b> is a dict-like object which reads and writes pandas using the high performance
HDF5 format using the <b>PyTables</b> library.

In [173]:
store = pd.HDFStore("store.h5")
print(store)

<class 'pandas.io.pytables.HDFStore'>
File path: store.h5



In [174]:
# Add objects to the file, same way as adding key/value pairs to a dict.
index = pd.date_range("1/1/2000", periods=8)
s = pd.Series(np.random.randn(5), index=["a", "b", "c", "d", "e"])
df = pd.DataFrame(np.random.randn(8, 3), index=index, columns=["A", "B", "C"])

store["s"] = s
store["df"] = df

store

<class 'pandas.io.pytables.HDFStore'>
File path: store.h5

In [175]:
store["df"]

,A,B,C
2000-01-01,-0.938651,0.219828,0.368594
2000-01-02,2.371900,-0.324893,0.324852
2000-01-03,0.550524,1.233317,-1.440167
2000-01-04,-1.214816,-1.208200,-0.198756
2000-01-05,-0.373763,1.089553,-1.143847
2000-01-06,1.405282,0.616418,0.933310
2000-01-07,0.746630,-1.470966,1.580739
2000-01-08,-0.097156,1.587288,0.150112


In [176]:
# Can also use attribute access
store.df

,A,B,C
2000-01-01,-0.938651,0.219828,0.368594
2000-01-02,2.371900,-0.324893,0.324852
2000-01-03,0.550524,1.233317,-1.440167
2000-01-04,-1.214816,-1.208200,-0.198756
2000-01-05,-0.373763,1.089553,-1.143847
2000-01-06,1.405282,0.616418,0.933310
2000-01-07,0.746630,-1.470966,1.580739
2000-01-08,-0.097156,1.587288,0.150112


In [177]:
# Delete an object in the store
del store["df"]
store

<class 'pandas.io.pytables.HDFStore'>
File path: store.h5

In [178]:
store.close()
store

<class 'pandas.io.pytables.HDFStore'>
File path: store.h5

In [179]:
store.is_open

False

## Read/Write API

In [180]:
df_tl = pd.DataFrame({"A": list(range(5)), "B": list(range(5))})
df_tl.to_hdf("store_tl.h5", key="table", append=True)

In [181]:
pd.read_hdf("store_tl.h5", "table", where=["index>2"])

,A,B
3,3,3
4,4,4
3,3,3
4,4,4


In [182]:
df_with_missing = pd.DataFrame({"col1": [0, np.nan, 2], "col2": [1, np.nan, np.nan]})
df_with_missing

,col1,col2
0,0.0,1.0
1,NaN,NaN
2,2.0,NaN


In [183]:
df_with_missing.to_hdf("file.h5", key="df_with_missing", format="table", mode="w")

In [184]:
pd.read_hdf("file.h5", "df_with_missing")

,col1,col2
0,0.0,1.0
1,NaN,NaN
2,2.0,NaN


In [185]:
df_with_missing.to_hdf("file.h5", key="df_with_missing", format="table", mode="w", dropna=True)

In [186]:
pd.read_hdf("file.h5", "df_with_missing")

,col1,col2
0,0.0,1.0
2,2.0,NaN


## Fixed Format

## Table Format

The <b>table</b> format is different from the standard <b>fixed</b> format.<br/>
With a <b>table</b> addition, delete and query operations are supported.<br/>
Also, it can be appended to another one.

In [187]:
store = pd.HDFStore("store.h5")
df1 = df[0:4]
df2 = df[4:]

In [188]:
# append data (creates a table automatically)
store.append("df", df1)
store.append("df", df2)

store

<class 'pandas.io.pytables.HDFStore'>
File path: store.h5

In [189]:
# select the entire object
store.select("df")

,A,B,C
2000-01-01,-0.938651,0.219828,0.368594
2000-01-02,2.371900,-0.324893,0.324852
2000-01-03,0.550524,1.233317,-1.440167
2000-01-04,-1.214816,-1.208200,-0.198756
2000-01-05,-0.373763,1.089553,-1.143847
2000-01-06,1.405282,0.616418,0.933310
2000-01-07,0.746630,-1.470966,1.580739
2000-01-08,-0.097156,1.587288,0.150112


In [190]:
# type of stored data
store.root.df._v_attrs.pandas_type

np.str_('frame_table')

## Hierarchical Keys

In [191]:
store.put("foo/bar/bah", df)
store.append("food/orange", df)
store.append("food/apple", df)

store

<class 'pandas.io.pytables.HDFStore'>
File path: store.h5

In [192]:
# return a list of keys
store.keys()

['/df', '/s', '/food/apple', '/food/orange', '/foo/bar/bah']

In [193]:
# remove all nodes under this level
store.remove("food")

In [194]:
for path, subgroups, subkeys in store.walk():
    for subgroup in subgroups:
        print(f"GROUP: {path}/{subgroup}")
    for subkey in subkeys:
        key = "/".join([path, subkey])
        print(f"KEY: {key}")
        print(store.get(key))

GROUP: /foo
KEY: /s
a   -0.528379
b    1.017217
c    0.490751
d    0.988934
e   -0.088686
dtype: float64
KEY: /df
                   A         B         C
2000-01-01 -0.938651  0.219828  0.368594
2000-01-02  2.371900 -0.324893  0.324852
2000-01-03  0.550524  1.233317 -1.440167
2000-01-04 -1.214816 -1.208200 -0.198756
2000-01-05 -0.373763  1.089553 -1.143847
2000-01-06  1.405282  0.616418  0.933310
2000-01-07  0.746630 -1.470966  1.580739
2000-01-08 -0.097156  1.587288  0.150112
GROUP: /foo/bar
KEY: /foo/bar/bah
                   A         B         C
2000-01-01 -0.938651  0.219828  0.368594
2000-01-02  2.371900 -0.324893  0.324852
2000-01-03  0.550524  1.233317 -1.440167
2000-01-04 -1.214816 -1.208200 -0.198756
2000-01-05 -0.373763  1.089553 -1.143847
2000-01-06  1.405282  0.616418  0.933310
2000-01-07  0.746630 -1.470966  1.580739
2000-01-08 -0.097156  1.587288  0.150112


## Storing Types
### Storing mixed types in a table

In [195]:
df_mixed = pd.DataFrame(
    {
        "A": np.random.randn(8),
        "B": np.random.randn(8),
        "C": np.array(np.random.randn(8), dtype="float32"),
        "string": "string",
        "int": 1,
        "bool": True,
        "datetime64": pd.Timestamp("20010102"),
    },
    index=list(range(8)),
)

In [196]:
df_mixed.loc[df_mixed.index[3:5], ["A", "B", "string", "datetime64"]] = np.nan

In [197]:
df_mixed

,A,B,C,string,int,bool,datetime64
0,0.299125,1.051205,-0.075895,string,1,True,2001-01-02
1,-0.889657,-1.790713,-0.326434,string,1,True,2001-01-02
2,-1.042957,-0.560826,-1.983762,string,1,True,2001-01-02
3,NaN,NaN,0.764824,NaN,1,True,NaT
4,NaN,NaN,-0.013560,NaN,1,True,NaT
5,1.108666,-0.873948,-1.379917,string,1,True,2001-01-02
6,-0.781491,-0.831029,-0.458024,string,1,True,2001-01-02
7,1.278225,-0.187035,-0.305238,string,1,True,2001-01-02


In [198]:
store.append("df_mixed", df_mixed, min_itemsize={"values": 50})
df_mixed1 = store.select("df_mixed")
df_mixed1

,A,B,C,string,int,bool,datetime64
0,0.299125,1.051205,-0.075895,string,1,True,1970-01-01 00:00:00.978393600
1,-0.889657,-1.790713,-0.326434,string,1,True,1970-01-01 00:00:00.978393600
2,-1.042957,-0.560826,-1.983762,string,1,True,1970-01-01 00:00:00.978393600
3,NaN,NaN,0.764824,NaN,1,True,NaT
4,NaN,NaN,-0.013560,NaN,1,True,NaT
5,1.108666,-0.873948,-1.379917,string,1,True,1970-01-01 00:00:00.978393600
6,-0.781491,-0.831029,-0.458024,string,1,True,1970-01-01 00:00:00.978393600
7,1.278225,-0.187035,-0.305238,string,1,True,1970-01-01 00:00:00.978393600


In [199]:
df_mixed1.dtypes.value_counts()

float64           2
float32           1
object            1
int64             1
bool              1
datetime64[ns]    1
Name: count, dtype: int64

## Qyerying
### Querying a Table

In [200]:
dfq = pd.DataFrame(
    np.random.randn(10, 4),
    columns=list("ABCD"),
    index=pd.date_range("20130101", periods=10),
)
store.append("dfq", dfq, format="table", data_columns=True)

In [202]:
store.select("dfq", "index>pd.Timestamp('20130104') & columns=['A', 'B']")

,A,B
2013-01-05,-0.450368,-0.767689
2013-01-06,0.366652,0.642717
2013-01-07,-1.125554,-0.274171
2013-01-08,0.147961,0.102974
2013-01-09,-1.001017,-1.743194
2013-01-10,1.399631,0.426891


# SQL Queries

In [ ]:
import adbc_driver_sqlite.dbapi as sqlite_dbapi

# Create the connection
with sqlite_dbapi.connect("sqlite:///:memory") as conn:
    df=pd.read_sql_table("data",conn)